In [1]:
"""
PV Module Single-Diode Simulation from EL-Extracted Rs and J0
=============================================================
Uses pvlib to simulate I-V curves from Rajput-method parameters
and overlays with measured flash I-V data.

Unit conversions (pixel → physical):
  I0 [A]  = J0_module_avg [A/pixel] × N_pixels_per_cell
  Rs [Ω]  = Rs_module_total [Ω·pixel] / N_pixels_per_cell
  Rsh [Ω] = N_cells × Rsh_cell  (Rsh_cell assumed 1000 Ω)
  nNsVth  = n × N_cells × kT/q  (n=1, T=25°C → kT/q=0.02585 V)
"""

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pvlib
import traceback
import time


# ──────────────────────────────────────────────────────────
# 1. MERGE module_summary with AnonDB
# ──────────────────────────────────────────────────────────
def prepare_merged_dataset(
    module_summary_csv,
    anondb_csv,
    output_csv="merged_for_simulation.csv"
):
    """
    Merge module_summary (Rs, J0) with AnonDB (Isc, Voc, IVPath, etc.)
    """
    mod = pd.read_csv(module_summary_csv)
    anon = pd.read_csv(anondb_csv)

    # only OK modules
    mod = mod[mod["status"] == "ok"].copy()

    # extract module base name from AnonDB ELLPath
    anon["ell_base"] = anon["ELLPath"].apply(
        lambda x: str(x).replace(".tiff", "").replace("_wrp", "").split("/")[-1].rsplit("_", 1)[0]
        if pd.notna(x) else ""
    )

    merged = mod.merge(anon, left_on="module", right_on="ell_base", how="inner")

    # compute physical parameters
    merged["N_pix_per_cell"] = merged["vals_size"] / merged["n_cells"]
    merged["I0_A"]   = merged["J0_module_avg"] * merged["N_pix_per_cell"]
    merged["Rs_Ohm"] = merged["Rs_module_total"] / merged["N_pix_per_cell"]
    merged["Rsh_Ohm"] = merged["n_cells"] * 1000.0   # 1000 Ω per cell
    merged["nNsVth"] = 1.0 * merged["n_cells"] * 0.02585  # n=1, T=25°C
    merged["IL_A"]   = merged["Isc_(A)"]

    merged.to_csv(output_csv, index=False)
    print(f"Merged dataset: {len(merged)} modules → {output_csv}")
    return merged


# ──────────────────────────────────────────────────────────
# 2. SIMULATE single-diode for all modules
# ──────────────────────────────────────────────────────────
def simulate_all_modules(
    merged_df,
    output_csv="simulated_iv_results.csv"
):
    """
    Run pvlib.pvsystem.singlediode for each module.
    Returns DataFrame with simulated + measured performance params.
    """
    results = []

    for idx, row in merged_df.iterrows():
        mod_name = row["module"]

        try:
            sim = pvlib.pvsystem.singlediode(
                photocurrent=row["IL_A"],
                saturation_current=row["I0_A"],
                resistance_series=row["Rs_Ohm"],
                resistance_shunt=row["Rsh_Ohm"],
                nNsVth=row["nNsVth"]
            )

            results.append({
                "module": mod_name,
                # --- measured ---
                "Isc_meas": row["Isc_(A)"],
                "Voc_meas": row["Voc_(V)"],
                "Imp_meas": row["Imp_(A)"],
                "Vmp_meas": row["Vmp_(V)"],
                "Pmp_meas": row["Pmp_(W)"],
                "FF_meas":  row["FF_(percent)"],
                # --- simulated ---
                "Isc_sim": sim["i_sc"],
                "Voc_sim": sim["v_oc"],
                "Imp_sim": sim["i_mp"],
                "Vmp_sim": sim["v_mp"],
                "Pmp_sim": sim["p_mp"],
                "FF_sim":  sim["p_mp"] / (sim["i_sc"] * sim["v_oc"]) * 100,
                # --- input parameters ---
                "I0_A":    row["I0_A"],
                "Rs_Ohm":  row["Rs_Ohm"],
                "Rsh_Ohm": row["Rsh_Ohm"],
                "nNsVth":  row["nNsVth"],
                "IL_A":    row["IL_A"],
                # --- errors ---
                "Voc_err_pct": (sim["v_oc"] - row["Voc_(V)"]) / row["Voc_(V)"] * 100,
                "Pmp_err_pct": (sim["p_mp"] - row["Pmp_(W)"]) / row["Pmp_(W)"] * 100,
                "FF_err_pct":  (sim["p_mp"] / (sim["i_sc"] * sim["v_oc"]) * 100 - row["FF_(percent)"]) / row["FF_(percent)"] * 100,
                # --- paths ---
                "IVPath": row.get("IVPath", ""),
                "status": "ok"
            })

        except Exception as e:
            print(f"  ERROR {mod_name}: {e}")
            results.append({
                "module": mod_name,
                "status": str(e)
            })

    results_df = pd.DataFrame(results)
    results_df.to_csv(output_csv, index=False)

    ok = results_df[results_df["status"] == "ok"]
    print(f"\nSimulation complete: {len(ok)}/{len(results_df)} OK → {output_csv}")
    if len(ok) > 0:
        print(f"\n  Voc error:  mean={ok['Voc_err_pct'].mean():.2f}%, std={ok['Voc_err_pct'].std():.2f}%")
        print(f"  Pmp error:  mean={ok['Pmp_err_pct'].mean():.2f}%, std={ok['Pmp_err_pct'].std():.2f}%")
        print(f"  FF  error:  mean={ok['FF_err_pct'].mean():.2f}%, std={ok['FF_err_pct'].std():.2f}%")

    return results_df


# ──────────────────────────────────────────────────────────
# 3. LOAD measured IV curve from CSV
# ──────────────────────────────────────────────────────────
def load_measured_iv(iv_path):
    """
    Load measured I-V data from CSV.
    Tries common column name patterns.
    """
    df = pd.read_csv(iv_path)

    # try to find voltage and current columns
    v_col = None
    i_col = None
    for c in df.columns:
        cl = c.lower().strip()
        if cl in ("voltage", "v", "voltage (v)", "voltage_(v)", "v(v)"):
            v_col = c
        elif cl in ("current", "i", "current (a)", "current_(a)", "i(a)"):
            i_col = c

    # fallback: first two numeric columns
    if v_col is None or i_col is None:
        num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
        if len(num_cols) >= 2:
            v_col, i_col = num_cols[0], num_cols[1]
        else:
            raise ValueError(f"Cannot identify V/I columns in {iv_path}: {df.columns.tolist()}")

    return df[v_col].values, df[i_col].values


# ──────────────────────────────────────────────────────────
# 4. GENERATE simulated IV curve points
# ──────────────────────────────────────────────────────────
def generate_simulated_iv(row, n_points=300):
    """
    Generate V, I arrays for the simulated IV curve.
    """
    v_oc = pvlib.pvsystem.singlediode(
        photocurrent=row["IL_A"],
        saturation_current=row["I0_A"],
        resistance_series=row["Rs_Ohm"],
        resistance_shunt=row["Rsh_Ohm"],
        nNsVth=row["nNsVth"]
    )["v_oc"]

    v_arr = np.linspace(0, v_oc * 1.02, n_points)
    i_arr = pvlib.pvsystem.i_from_v(
        voltage=v_arr,
        photocurrent=row["IL_A"],
        saturation_current=row["I0_A"],
        resistance_series=row["Rs_Ohm"],
        resistance_shunt=row["Rsh_Ohm"],
        nNsVth=row["nNsVth"]
    )
    # clip negative currents
    i_arr = np.maximum(i_arr, 0)
    return v_arr, i_arr


# ──────────────────────────────────────────────────────────
# 5. PLOT: overlay simulated vs measured IV
# ──────────────────────────────────────────────────────────
def plot_iv_overlay_single(row, base_dir=".", save_path=None):
    """
    Plot a single module: simulated IV + measured IV overlaid.
    """
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # --- simulated ---
    v_sim, i_sim = generate_simulated_iv(row)
    p_sim = v_sim * i_sim

    ax1.plot(v_sim, i_sim, "b-", linewidth=2, label="Simulated (EL)")
    ax2.plot(v_sim, p_sim, "b-", linewidth=2, label="Simulated (EL)")

    # --- measured ---
    iv_path = os.path.normpath(os.path.join(base_dir, row["IVPath"]))
    if os.path.isfile(iv_path):
        try:
            v_meas, i_meas = load_measured_iv(iv_path)
            p_meas = v_meas * i_meas
            ax1.plot(v_meas, i_meas, "r--", linewidth=1.5, label="Measured (Flash)")
            ax2.plot(v_meas, p_meas, "r--", linewidth=1.5, label="Measured (Flash)")
        except Exception as e:
            print(f"  Could not load measured IV: {e}")
    else:
        print(f"  IV file not found: {iv_path}")

    # --- MPP markers ---
    ax1.plot(row["Vmp_sim"], row["Imp_sim"], "bs", markersize=8, label=f"MPP sim ({row['Pmp_sim']:.1f}W)")
    ax2.plot(row["Vmp_sim"], row["Pmp_sim"], "bs", markersize=8)

    if pd.notna(row.get("Vmp_meas")) and pd.notna(row.get("Imp_meas")):
        ax1.plot(row["Vmp_meas"], row["Imp_meas"], "r^", markersize=8, label=f"MPP meas ({row['Pmp_meas']:.1f}W)")
        ax2.plot(row["Vmp_meas"], row["Pmp_meas"], "r^", markersize=8)

    mod = row["module"]
    ax1.set_xlabel("Voltage (V)")
    ax1.set_ylabel("Current (A)")
    ax1.set_title(f"{mod} — I-V Curve")
    ax1.legend(fontsize=9)
    ax1.grid(True, alpha=0.3)
    ax1.set_xlim(0, None)
    ax1.set_ylim(0, None)

    ax2.set_xlabel("Voltage (V)")
    ax2.set_ylabel("Power (W)")
    ax2.set_title(f"{mod} — P-V Curve")
    ax2.legend(fontsize=9)
    ax2.grid(True, alpha=0.3)
    ax2.set_xlim(0, None)
    ax2.set_ylim(0, None)

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        plt.close()
    else:
        plt.show()


def plot_iv_overlay_batch(
    results_df,
    base_dir=".",
    save_dir=None,
    max_plots=None
):
    """
    Plot IV overlays for all modules.
    If save_dir is set, saves PNG files instead of displaying.
    """
    ok = results_df[results_df["status"] == "ok"]
    if max_plots:
        ok = ok.head(max_plots)

    if save_dir:
        os.makedirs(save_dir, exist_ok=True)

    for idx, row in ok.iterrows():
        save_path = os.path.join(save_dir, f"{row['module']}_IV.png") if save_dir else None
        print(f"  Plotting {row['module']}...")
        plot_iv_overlay_single(row, base_dir=base_dir, save_path=save_path)

    print(f"Done! {len(ok)} plots generated.")


# ──────────────────────────────────────────────────────────
# 6. PLOT: summary comparison scatter plots
# ──────────────────────────────────────────────────────────
def plot_summary_scatter(results_df, save_path=None):
    """
    4-panel scatter: Voc, Isc, Pmp, FF (measured vs simulated)
    """
    ok = results_df[results_df["status"] == "ok"].copy()

    fig, axes = plt.subplots(2, 2, figsize=(12, 10))

    params = [
        ("Voc_meas", "Voc_sim", "Voc (V)", axes[0, 0]),
        ("Pmp_meas", "Pmp_sim", "Pmp (W)", axes[0, 1]),
        ("Imp_meas", "Imp_sim", "Imp (A)", axes[1, 0]),
        ("FF_meas",  "FF_sim",  "FF (%)",  axes[1, 1]),
    ]

    for meas_col, sim_col, label, ax in params:
        x = ok[meas_col]
        y = ok[sim_col]

        ax.scatter(x, y, alpha=0.5, s=20, edgecolors="k", linewidth=0.3)

        # 1:1 line
        lo = min(x.min(), y.min()) * 0.98
        hi = max(x.max(), y.max()) * 1.02
        ax.plot([lo, hi], [lo, hi], "r--", linewidth=1, label="1:1")

        # stats
        from scipy import stats
        slope, intercept, r, p, se = stats.linregress(x, y)
        mae = np.mean(np.abs(y - x))
        ax.set_title(f"{label}  |  R²={r**2:.3f}  MAE={mae:.2f}", fontsize=11)
        ax.set_xlabel(f"Measured {label}")
        ax.set_ylabel(f"Simulated {label}")
        ax.legend()
        ax.grid(True, alpha=0.3)

    plt.suptitle("Simulated (EL single-diode) vs Measured (Flash I-V)", fontsize=13, y=1.01)
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        plt.close()
    else:
        plt.show()


# ──────────────────────────────────────────────────────────
# MAIN: run everything
# ──────────────────────────────────────────────────────────
if __name__ == "__main__":

    BASE_DIR = r"C:\Users\Ghozy Abror\OneDrive - Institut Teknologi Bandung\Karirku\UNSW\Thesis\Coding"

    # Step 1: merge datasets
    merged = prepare_merged_dataset(
        module_summary_csv="module_summary.csv",
        anondb_csv="AnonDB_zub_60.csv",
        output_csv="merged_for_simulation.csv"
    )

    # Step 2: simulate all modules
    results = simulate_all_modules(
        merged,
        output_csv="simulated_iv_results.csv"
    )

    # Step 3: summary scatter plots
    plot_summary_scatter(results, save_path="sim_vs_meas_scatter.png")

    # Step 4: overlay IV curves (first 10, or all)
    plot_iv_overlay_batch(
        results,
        base_dir=BASE_DIR,
        save_dir="IV_overlay_plots",
        max_plots=10          # set to None for all
    )

Merged dataset: 274 modules → merged_for_simulation.csv

Simulation complete: 274/274 OK → simulated_iv_results.csv

  Voc error:  mean=0.05%, std=1.48%
  Pmp error:  mean=2.72%, std=3.65%
  FF  error:  mean=2.66%, std=3.15%
  Plotting 3855_17_1_01302020...
  Plotting 3857_17_1_02222021...
  Plotting 3857_17_1_01302020...
  Plotting 3858_17_1_02222021...
  Plotting 3859_17_1_02222021...
  Plotting 3859_17_1_01302020...
  Plotting 3865_17_1_02222021...
  Plotting 3865_17_1_01302020...
  Plotting 3866_17_1_02222021...
  Plotting 3866_17_1_01302020...
Done! 10 plots generated.
